# 02 · Modelagem: comparação de comitês

**Esqueleto para o responsável pelo código.** As células abaixo já carregam os dados, a validação cruzada e o pré-processamento, e calculam a linha de base. Cada seção `TODO` corresponde a um item exigido no enunciado (seções 3 e 4).

Regras de protocolo que a base de código já garante:

| Regra (enunciado §3) | Como está garantido |
|---|---|
| Teste separado no início e usado uma vez | `dataset/splits/test.csv`, só lido por `load_test()` na última seção |
| Pré-processamento dentro de pipeline | `build_preprocessor()` devolve um transformador **não ajustado**; sempre usar `make_pipeline(build_preprocessor(), modelo)` |
| Mesmas divisões para todas as alternativas | `cv = get_cv()`, o mesmo objeto passado a todo `cross_validate` / `GridSearchCV` |
| Sementes fixas | `C.SEED = 42`; passar `random_state=C.SEED` a todo modelo estocástico |

In [1]:
# Torna o pacote `tde2` (em ../src) importável, rodando local ou no Colab após `git clone`.
import sys
from pathlib import Path

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "tde2").is_dir())
sys.path.insert(0, str(_root / "src"))

In [2]:
import warnings

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline

from tde2 import config as C
from tde2.data import get_cv, load_train
from tde2.preprocessing import build_preprocessor

X_train, y_train = load_train()
cv = get_cv()
print(X_train.shape, y_train.value_counts().to_dict())

(612, 11) {1: 339, 0: 273}


In [3]:
def evaluate(name, estimator, results):
    """Validação cruzada com as mesmas dobras e métricas para qualquer alternativa."""
    r = cross_validate(estimator, X_train, y_train, cv=cv, scoring=C.SCORING,
                       return_train_score=False)  # sem n_jobs: tempos de fit/predição comparáveis
    row = {"modelo": name}
    for m in C.SCORING:
        row[f"{m}_media"] = r[f"test_{m}"].mean()
        row[f"{m}_dp"] = r[f"test_{m}"].std()
    row["fit_time_s"] = r["fit_time"].mean()
    row["score_time_s"] = r["score_time"].mean()
    results.append(row)
    return r

results = []

## 1. Linha de base

In [4]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # roc_auc/f1 indefinidos para classe nunca predita
    evaluate("Baseline: classe majoritária",
             make_pipeline(build_preprocessor(), DummyClassifier(strategy="most_frequent")),
             results)
pd.DataFrame(results).set_index("modelo").round(3)

,balanced_accuracy_media,balanced_accuracy_dp,f1_macro_media,f1_macro_dp,recall_doenca_media,recall_doenca_dp,roc_auc_media,roc_auc_dp,fit_time_s,score_time_s
modelo,,,,,,,,,,
Baseline: classe majoritária,0.5,0.0,0.356,0.002,1.0,0.0,0.5,0.0,0.008,0.016


## 2. Membros individuais — TODO

Avaliar cada membro **isoladamente**, com `evaluate(...)`, antes de montar os comitês. Os mesmos membros entram no comitê heterogêneo.

Sugestão de membros com vieses indutivos diferentes (diversidade): regressão logística, KNN, SVM, árvore de decisão, Naive Bayes.

## 3. Comitê heterogêneo (Aula 6) — TODO

`VotingClassifier` (hard/soft) ou `StackingClassifier`, com a **regra de fusão declarada** no relatório.

## 4. Comitê homogêneo, redução de variância (Aula 9) — TODO

`BaggingClassifier` ou `RandomForestClassifier`, com membro-base avaliado individualmente.

## 5. Comitê homogêneo, redução de viés (Aula 9) — TODO

`AdaBoostClassifier` / `GradientBoostingClassifier` / `HistGradientBoostingClassifier`, com membro-base avaliado individualmente.

## 6. Ajuste de hiperparâmetros com esforço equivalente — TODO

Mesmo orçamento para todas as alternativas (ex.: `RandomizedSearchCV(n_iter=N, cv=cv, scoring=C.PRIMARY_METRIC, random_state=C.SEED)`). Documentar grades e valores escolhidos.

## 7. Diversidade, erros por classe e custo — TODO

- Diversidade: discordância par a par / estatística Q de Yule sobre `cross_val_predict(..., cv=cv)`.
- Matriz de confusão por alternativa (predições fora da dobra).
- Custo: `fit_time` / `score_time` já coletados por `evaluate`.

## 8. Avaliação final no conjunto de teste — executar UMA vez

Só depois que **todas** as configurações estiverem congeladas. Descomente quando chegar a hora.

In [5]:
# from tde2.data import load_test
# X_test, y_test = load_test()
# for name, model in final_models.items():
#     model.fit(X_train, y_train)
#     ...